# Fire Detection — YOLOv11 Training on Google Colab
**Author:** Alaaeddine Bouchamla — ENISO M1 Telecommunications

**Runtime:** GPU (T4 recommended) → Runtime > Change runtime type > T4 GPU

This notebook:
1. Downloads the D-Fire dataset from Kaggle
2. Trains YOLOv11n and YOLOv11x on fire/smoke detection
3. Evaluates on the test set and prints all paper metrics
4. Exports YOLOv11n to ONNX for edge deployment

## 0. Setup

In [ ]:
# Install dependencies
!pip install ultralytics kaggle -q
import ultralytics
ultralytics.checks()

## 1. Download D-Fire Dataset from Kaggle

You need a Kaggle account + API key.
1. Go to https://www.kaggle.com/settings
2. Scroll to "API" → click **Create New Token** → downloads `kaggle.json`
3. Upload that file below.

In [ ]:
from google.colab import files
print('Upload your kaggle.json file:')
files.upload()

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

# Download D-Fire (YOLO-ready version)
!kaggle datasets download -d sayedgamal99/smoke-fire-detection-yolo -p /content/
!unzip -q /content/smoke-fire-detection-yolo.zip -d /content/d-fire
!ls /content/d-fire

In [ ]:
# Inspect dataset structure
import os
for root, dirs, files_list in os.walk('/content/d-fire'):
    level = root.replace('/content/d-fire', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 2:
        subindent = ' ' * 2 * (level + 1)
        for f in files_list[:3]:
            print(f'{subindent}{f}')
        if len(files_list) > 3:
            print(f'{subindent}... ({len(files_list)} files total)')

## 2. Create Dataset Config

In [ ]:
import yaml, os

# Auto-detect train/val/test paths
base = '/content/d-fire'
splits = {}
for split in ['train', 'val', 'test', 'valid']:
    p = os.path.join(base, split, 'images')
    if os.path.exists(p):
        splits[split if split != 'valid' else 'val'] = p

dataset_config = {
    'path': base,
    'train': splits.get('train', 'train/images'),
    'val': splits.get('val', 'val/images'),
    'test': splits.get('test', 'test/images'),
    'nc': 2,
    'names': {0: 'fire', 1: 'smoke'}
}

config_path = '/content/fire-dataset.yaml'
with open(config_path, 'w') as f:
    yaml.dump(dataset_config, f)

print('Dataset config:')
print(yaml.dump(dataset_config))

## 3. Train YOLOv11n (Edge Model)

In [ ]:
from ultralytics import YOLO

model_n = YOLO('yolo11n.pt')

results_n = model_n.train(
    data=config_path,
    epochs=100,
    batch=16,
    imgsz=640,
    device=0,
    name='fire-yolo11n',
    optimizer='AdamW',
    lr0=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=1.0,
    degrees=10.0,
    plots=True,
    save=True,
)

print(f'YOLOv11n best weights: {results_n.save_dir}/weights/best.pt')

## 4. Train YOLOv11x (Cloud Baseline)

In [ ]:
model_x = YOLO('yolo11x.pt')

results_x = model_x.train(
    data=config_path,
    epochs=100,
    batch=8,           # smaller batch — yolo11x is larger
    imgsz=640,
    device=0,
    name='fire-yolo11x',
    optimizer='AdamW',
    lr0=0.01,
    cos_lr=True,
    warmup_epochs=3,
    mosaic=1.0,
    plots=True,
    save=True,
)

print(f'YOLOv11x best weights: {results_x.save_dir}/weights/best.pt')

## 5. Evaluate Both Models — Paper Metrics

In [ ]:
import json

def evaluate_model(weights_path, label):
    model = YOLO(weights_path)
    metrics = model.val(data=config_path, split='test', plots=True)
    result = {
        'model': label,
        'precision': round(float(metrics.box.mp), 4),
        'recall': round(float(metrics.box.mr), 4),
        'mAP50': round(float(metrics.box.map50), 4),
        'mAP50_95': round(float(metrics.box.map), 4),
        'per_class': {
            name: {
                'precision': round(float(p), 4),
                'recall': round(float(r), 4),
                'mAP50': round(float(ap50), 4),
            }
            for name, p, r, ap50 in zip(
                metrics.names.values(),
                metrics.box.p, metrics.box.r, metrics.box.ap50
            )
        }
    }
    return result

metrics_n = evaluate_model(f'{results_n.save_dir}/weights/best.pt', 'YOLOv11n (edge, FP32)')
metrics_x = evaluate_model(f'{results_x.save_dir}/weights/best.pt', 'YOLOv11x (cloud)')

all_metrics = [metrics_n, metrics_x]

print('\n══ PAPER METRICS ════════════════════════════════')
print(f'{"Model":<30} {"P":>6} {"R":>6} {"mAP50":>7} {"mAP50-95":>10}')
print('-' * 65)
for m in all_metrics:
    print(f"{m['model']:<30} {m['precision']:>6.4f} {m['recall']:>6.4f} {m['mAP50']:>7.4f} {m['mAP50_95']:>10.4f}")
print('━' * 65)
print('\nPer-class breakdown:')
for m in all_metrics:
    print(f"\n{m['model']}:")
    for cls, vals in m['per_class'].items():
        print(f"  {cls:10s}: mAP50={vals['mAP50']:.4f}  P={vals['precision']:.4f}  R={vals['recall']:.4f}")

with open('/content/metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print('\nSaved to /content/metrics.json')

## 6. Export YOLOv11n to ONNX (Edge Deployment)

In [ ]:
import time
import numpy as np
import onnxruntime as ort

model_n_best = YOLO(f'{results_n.save_dir}/weights/best.pt')
onnx_path = model_n_best.export(format='onnx', imgsz=640, simplify=True)
print(f'ONNX exported: {onnx_path}')

# Benchmark ONNX on CPU (simulates edge device)
session = ort.InferenceSession(str(onnx_path), providers=['CPUExecutionProvider'])
dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)
input_name = session.get_inputs()[0].name

for _ in range(5):  # warm-up
    session.run(None, {input_name: dummy})

times = []
for _ in range(50):
    t0 = time.perf_counter()
    session.run(None, {input_name: dummy})
    times.append((time.perf_counter() - t0) * 1000)

avg_ms = sum(times) / len(times)
print(f'\nONNX CPU inference latency: {avg_ms:.1f} ms (avg over 50 runs)')
print('NOTE: Real edge hardware (Jetson Nano) will be faster with TensorRT INT8')

print(f'\n══ LATENCY SUMMARY (fill into paper.md) ══')
print(f'Edge inference (ONNX CPU):  {avg_ms:.1f} ms')
print(f'Event encoding:             ~2 ms')
print(f'4G/LTE uplink:              ~30 ms (estimated)')
print(f'Cloud DB write + push:      ~50 ms (estimated)')
print(f'Dashboard render:           ~100 ms (estimated)')
print(f'Total end-to-end:           ~{avg_ms + 2 + 30 + 50 + 100:.0f} ms (~{(avg_ms + 182)/1000:.2f} s)')

## 7. Download Results

In [ ]:
import shutil
from google.colab import files

# Package everything into a zip
shutil.make_archive('/content/fire_detection_results', 'zip',
                    '/content/runs')

# Also zip the ONNX model and metrics
os.makedirs('/content/export', exist_ok=True)
shutil.copy(str(onnx_path), '/content/export/yolo11n-fire.onnx')
shutil.copy('/content/metrics.json', '/content/export/metrics.json')
shutil.make_archive('/content/fire_model_export', 'zip', '/content/export')

print('Downloading training results (plots, weights)...')
files.download('/content/fire_detection_results.zip')
print('Downloading model export (ONNX + metrics)...')
files.download('/content/fire_model_export.zip')